In [1]:
from collections import Counter
import pandas as pd
from Bio import SeqIO

In [2]:
# collect CDS features from the GenBank file into a table
rows = []
for rec in SeqIO.parse('../../data/reference/mtdna_refseq.gb', 'genbank'):
    # species: prefer annotation, fallback to description
    species = rec.annotations.get('organism') or rec.description.split(',')[0]
    genus = species.split(' ')[0]
    # try to get phylum from source feature qualifiers; if not present, try taxonomy annotation

    tax = rec.annotations.get('taxonomy')

    for feat in rec.features:
        if feat.type == 'CDS':
            # gene name: prefer 'gene', then 'product', then 'locus_tag'
            gene = ''
            if 'gene' in feat.qualifiers:
                gene = feat.qualifiers['gene'][0]
            # elif 'product' in feat.qualifiers:
            #     gene = feat.qualifiers['product'][0]
            # elif 'locus_tag' in feat.qualifiers:
            #     gene = feat.qualifiers['locus_tag'][0]
            else:
                continue

            strand = '+' if feat.location.strand == 1 else ('-' if feat.location.strand == -1 else '')
            seq = str(feat.extract(rec.seq))  # nucleotide sequence of the CDS

            rows.append({
                'id': rec.id,
                'species': species,
                'genus': genus,
                'tax': tax,
                'gene name': gene,
                'strand': strand,
                'nt_sequence': seq,
                'aa_sequence': feat.qualifiers['translation'][0] if 'translation' in feat.qualifiers else ''
            })

df_cds = pd.DataFrame(rows)
df_cds.head()


,id,species,genus,tax,gene name,strand,nt_sequence,aa_sequence
0,NC_021392.1,Scomberomorus munroi x Scomberomorus semifasci...,Scomberomorus,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",CYTB,+,ATGGCAAGCCTACGAAAAACTCACCCACTACTAAAAATCGCAAACG...,MASLRKTHPLLKIANDALVDLPTPASISAWWNFGSLLGLCLISQIL...
1,NC_021392.1,Scomberomorus munroi x Scomberomorus semifasci...,Scomberomorus,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",ND1,+,ATGATCACCGCCCTACTAATTCACATCCTTAACCCCCTAGCCTTCA...,MITALLIHILNPLAFIVPVLLAVAFLTLIERKVLGYMQLRKGPNIV...
2,NC_021392.1,Scomberomorus munroi x Scomberomorus semifasci...,Scomberomorus,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",ND2,+,ATGAACCCATACATCTTAGCTACCCTACTATTTGGTCTAGGCCTAG...,MNPYILATLLFGLGLGTTITFASSHWLLAWMGLEMNTLAIIPLMAQ...
3,NC_021392.1,Scomberomorus munroi x Scomberomorus semifasci...,Scomberomorus,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",COX1,+,GTGGCAATCACACGCTGATTTTTCTCGACCAACCATAAAGACATTG...,MAITRWFFSTNHKDIGTLYLVFGAWAGMVGTALSLLIRAELSQPGA...
4,NC_021392.1,Scomberomorus munroi x Scomberomorus semifasci...,Scomberomorus,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",COX2,+,ATGGCCCATCCCACACAACTAGGATTTCAAGATGCAGCTTCACCTG...,MAHPTQLGFQDAASPVMEELLHFHDHALMIVFLISALVLYIIVAML...


In [3]:
df_species = df_cds[['id', 'species', 'genus', 'tax']].drop_duplicates('id').reset_index(drop=True)
df_species['kingdom'] = df_species['tax'].apply(lambda x: x[1] if len(x) > 1 else '')
df_species

,id,species,genus,tax,kingdom
0,NC_021392.1,Scomberomorus munroi x Scomberomorus semifasci...,Scomberomorus,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",Metazoa
1,NC_021391.1,Scomberomorus semifasciatus,Scomberomorus,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",Metazoa
2,NC_021390.1,Scomberomorus munroi,Scomberomorus,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",Metazoa
3,NC_062466.1,Diglossa brunneiventris,Diglossa,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",Metazoa
4,NC_030608.1,Dimidiochromis kiwinge,Dimidiochromis,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",Metazoa
...,...,...,...,...,...
150,NC_001808.1,Ceratotherium simum,Ceratotherium,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",Metazoa
151,NC_001700.1,Felis catus,Felis,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",Metazoa
152,NC_001602.1,Halichoerus grypus,Halichoerus,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",Metazoa
153,NC_001601.1,Balaenoptera musculus,Balaenoptera,"[Eukaryota, Metazoa, Chordata, Craniata, Verte...",Metazoa


In [4]:
# annotate using taxonkit
# cat ./data/species_list.txt | taxonkit name2taxid | taxonkit lineage --taxid-field 2 | taxonkit reformat -I 2 > ../../data/reference/species_lineages.txt

df_species['species'].drop_duplicates().to_csv('data/species_list.txt', index=False, header=False)


In [5]:
df_species['kingdom'].value_counts()

kingdom
Metazoa          137
Fungi             16
Viridiplantae      1
Sar                1
Name: count, dtype: int64

In [6]:
species_lineages = pd.read_csv('../../data/reference/species_lineages.txt', sep='\t', header=None, names=['species', 'taxid', 'raw_lineage', 'lineage'])
species_lineages['phylum'] = species_lineages['lineage'].apply(lambda x: x.split(';')[1])
species_lineages['genus'] = species_lineages['lineage'].apply(lambda x: x.split(';')[5])
species_lineages.head()


,species,taxid,raw_lineage,lineage,phylum,genus
0,Scomberomorus munroi x Scomberomorus semifasci...,1238137,cellular organisms;Eukaryota;Opisthokonta;Meta...,Eukaryota;Chordata;Actinopteri;Scombriformes;S...,Chordata,Scomberomorus
1,Scomberomorus semifasciatus,248138,cellular organisms;Eukaryota;Opisthokonta;Meta...,Eukaryota;Chordata;Actinopteri;Scombriformes;S...,Chordata,Scomberomorus
2,Scomberomorus munroi,248137,cellular organisms;Eukaryota;Opisthokonta;Meta...,Eukaryota;Chordata;Actinopteri;Scombriformes;S...,Chordata,Scomberomorus
3,Diglossa brunneiventris,547948,cellular organisms;Eukaryota;Opisthokonta;Meta...,Eukaryota;Chordata;Aves;Passeriformes;Thraupid...,Chordata,Diglossa
4,Dimidiochromis kiwinge,163619,cellular organisms;Eukaryota;Opisthokonta;Meta...,Eukaryota;Chordata;Actinopteri;Cichliformes;Ci...,Chordata,Dimidiochromis


In [7]:
species_lineages.query('phylum == "Mollusca"')['lineage'].unique()

<StringArray>
['Eukaryota;Mollusca;Bivalvia;Unionida;Margaritiferidae;Cumberlandia;Cumberlandia monodonta',
           'Eukaryota;Mollusca;Bivalvia;Unionida;Hyriidae;Echyridella;Echyridella menziesii',
                        'Eukaryota;Mollusca;Bivalvia;Unionida;Unionidae;Unio;Unio delphinus',
                       'Eukaryota;Mollusca;Bivalvia;Unionida;Unionidae;Lens;Lens contradens',
            'Eukaryota;Mollusca;Bivalvia;Unionida;Unionidae;Rectidens;Rectidens sumatrensis',
                'Eukaryota;Mollusca;Bivalvia;Unionida;Unionidae;Hyriopsis;Hyriopsis bialata',
                 'Eukaryota;Mollusca;Bivalvia;Unionida;Unionidae;Physunio;Physunio superbus',
                        'Eukaryota;Mollusca;Bivalvia;Unionida;Mutelidae;Mutela;Mutela dubia',
   'Eukaryota;Mollusca;Bivalvia;Trigoniida;Trigoniidae;Neotrigonia;Neotrigonia margaritacea',
 'Eukaryota;Mollusca;Gastropoda;Neogastropoda;Muricidae;Concholepas;Concholepas concholepas',
          'Eukaryota;Mollusca;Bivalvia;Mytilid

In [8]:
df_species_flt = df_species.merge(species_lineages[['genus', 'phylum']].drop_duplicates())
print(df_species_flt['phylum'].value_counts())

df_species_flt = df_species_flt.query("kingdom == 'Metazoa' & phylum in ['Mollusca', 'Arthropoda', 'Chordata']")

phylum
Chordata           65
Porifera           23
Mollusca           22
Arthropoda         15
Ascomycota         13
Nematoda            3
Echinodermata       3
Rotifera            2
Mucoromycota        2
Platyhelminthes     1
Streptophyta        1
Ctenophora          1
Ciliophora          1
Cnidaria            1
Hemichordata        1
Chytridiomycota     1
Name: count, dtype: int64


In [9]:
df_cds.aa_sequence.str.len().describe()

count    2250.000000
mean      289.247556
std       167.726403
min        15.000000
25%       144.000000
50%       261.000000
75%       388.000000
max      1712.000000
Name: aa_sequence, dtype: float64

In [10]:
df_species_flt.genus.value_counts().head(10)

genus
Pelophylax       6
Phylloscopus     5
Lycodon          4
Scomberomorus    3
Anopheles        3
Mus              3
Scincella        3
Rhipicephalus    2
Jacana           2
Triplophysa      2
Name: count, dtype: int64

In [11]:
df_cds_flt = df_cds.merge(
    df_species_flt.drop_duplicates('genus', keep='last')\
        .drop(['tax', 'genus', 'id'], axis=1), 
    on='species')

used_genes = df_cds_flt['gene name'].value_counts().head(13).index.values
df_cds_flt = df_cds_flt[df_cds_flt['gene name'].isin(used_genes)]

df_cds_flt.species.nunique()

75

In [12]:
df_cds_flt.phylum.value_counts()

phylum
Chordata      559
Mollusca      256
Arthropoda    156
Name: count, dtype: int64

In [13]:
df_cds_flt['gene name'].value_counts()

gene name
CYTB    75
ND1     75
ND2     75
ATP6    75
ND3     75
ND4L    75
ND4     75
ND5     75
ND6     75
COX1    74
COX2    74
ATP8    74
COX3    74
Name: count, dtype: int64

In [14]:
_ = df_cds_flt.groupby(['phylum', 'gene name'])['strand'].value_counts()\
    .unstack(1).fillna(0)
_.to_csv('data/gene_strand_counts_by_phylum.csv')
_

gene name          ATP6  ATP8  COX1  COX2  COX3  CYTB   ND1   ND2   ND3   ND4  \
phylum     strand                                                               
Arthropoda +       12.0  12.0  12.0  12.0  12.0  12.0   1.0  12.0  12.0   0.0   
           -        0.0   0.0   0.0   0.0   0.0   0.0  11.0   0.0   0.0  12.0   
Chordata   +       43.0  43.0  43.0  43.0  43.0  43.0  43.0  43.0  43.0  43.0   
           -        0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   
Mollusca   +        7.0   6.0   6.0   6.0   6.0  16.0  16.0  16.0   7.0   7.0   
           -       13.0  13.0  13.0  13.0  13.0   4.0   4.0   4.0  13.0  13.0   

gene name          ND4L   ND5   ND6  
phylum     strand                    
Arthropoda +        0.0   0.0  12.0  
           -       12.0  12.0   0.0  
Chordata   +       43.0  43.0   0.0  
           -        0.0   0.0  43.0  
Mollusca   +        7.0   7.0  16.0  
           -       13.0  13.0   4.0

In [15]:
dataset = df_cds_flt.query('`gene name` != "ATP9" & phylum in ["Mollusca", "Arthropoda", "Chordata"]')\
    .set_index(['phylum', 'species', 'gene name', 'strand'])\
        .aa_sequence.apply(Counter).apply(pd.Series).drop(['X', 'B'], axis=1).fillna(0)

# dataset = df_cds_flt.query('`gene name` != "ATP9" & phylum in ["Mollusca"]')\
#     .set_index(['phylum', 'species', 'gene name', 'strand'])\
#         .aa_sequence.apply(Counter).apply(pd.Series).drop(['X'], axis=1).fillna(0)

dataset

M     A     S      L  \
phylum   species               gene name strand                            
Chordata Scomberomorus munroi  CYTB      +       11.0  28.0  23.0   59.0   
                               ND1       +       10.0  38.0  18.0   62.0   
                               ND2       +       17.0  35.0  26.0   81.0   
                               COX1      +       24.0  48.0  27.0   62.0   
                               COX2      +       11.0  19.0  17.0   28.0   
...                                               ...   ...   ...    ...   
         Balaenoptera musculus ND4L      +       11.0  10.0   8.0   23.0   
                               ND4       +       33.0  33.0  37.0   98.0   
                               ND5       +       30.0  40.0  41.0  101.0   
                               ND6       -        5.0   8.0  15.0   18.0   
                               CYTB      +       16.0  25.0  21.0   59.0   

                                                    R     K     T     H     P  \
phylum   species               gene name strand                                 
Chordata Scomberomorus munroi  CYTB      +        8.0   9.0  22.0  11.0  24.0   
                               ND1       +        8.0   7.0  20.0   4.0  24.0   
                               ND2       +        4.0   9.0  43.0   7.0  25.0   
                               COX1      +        9.0   8.0  37.0  20.0  28.0   
                               COX2      +        6.0   4.0  10.0  10.0  14.0   
...                                               ...   ...   ...   ...   ...   
         Balaenoptera musculus ND4L      +        1.0   0.0   5.0   3.0   1.0   
                               ND4       +       10.0  12.0  33.0  14.0  24.0   
                               ND5       +        8.0  22.0  61.0  13.0  32.0   
                               ND6       -        1.0   4.0   8.0   0.0   3.0   
                               CYTB      +        9.0  10.0  28.0  13.0  23.0   

                                                    I     N     D     W     F  \
phylum   species               gene name strand                                 
Chordata Scomberomorus munroi  CYTB      +       35.0  19.0  10.0  12.0  33.0   
                               ND1       +       27.0  12.0   4.0   8.0  17.0   
                               ND2       +       17.0  13.0   2.0  11.0   9.0   
                               COX1      +       40.0  19.0  14.0  17.0  42.0   
                               COX2      +       21.0   5.0  13.0   5.0   9.0   
...                                               ...   ...   ...   ...   ...   
         Balaenoptera musculus ND4L      +        5.0   5.0   1.0   0.0   4.0   
                               ND4       +       42.0  22.0   3.0  13.0  16.0   
                               ND5       +       55.0  35.0  12.0  12.0  46.0   
                               ND6       -       17.0   2.0   3.0   4.0  12.0   
                               CYTB      +       36.0  14.0  12.0  12.0  26.0   

                                                    G    C     Q     Y     E  \
phylum   species               gene name strand                                
Chordata Scomberomorus munroi  CYTB      +       26.0  3.0   6.0  13.0   8.0   
                               ND1       +       18.0  0.0   7.0  12.0  11.0   
                               ND2       +       17.0  1.0  12.0   7.0   6.0   
                               COX1      +       46.0  1.0   7.0  18.0  12.0   
                               COX2      +        8.0  2.0   9.0   9.0  12.0   
...                                               ...  ...   ...   ...   ...   
         Balaenoptera musculus ND4L      +        4.0  2.0   2.0   4.0   1.0   
                               ND4       +       17.0  3.0  10.0  15.0   9.0   
                               ND5       +       27.0  7.0  21.0  16.0  11.0   
                               ND6       -       27.0  1.0   1.0   9.0   8.

In [16]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import seaborn as sns
from aminoacidshift import amino_acid_codes

# PCA plot of `dataset` (uses MultiIndex levels 'gene name' and 'strand' for hue/style)
import matplotlib.pyplot as plt

# prepare data
df = dataset.rename(columns=amino_acid_codes).reset_index()  # brings 'gene name' and 'strand' into columns
features = list(amino_acid_codes.values())[:-1]  # amino acid features
X = df[features].values

# scale and run PCA
X_scaled = StandardScaler().fit_transform(X)
pca = PCA(n_components=2, random_state=0)
pcs = pca.fit_transform(X_scaled)

# build plotting frame
plot_df = df[['gene name', 'strand', 'phylum']].copy()
plot_df.columns = ['Ген', 'Цепь', 'phylum']
plot_df['PC1'], plot_df['PC2'] = pcs[:, 0], pcs[:, 1]

# plot
sns.set(style='whitegrid')
plt.figure(figsize=(8, 6))
ax = sns.scatterplot(
    data=plot_df,
    x='PC1', y='PC2',
    hue='Ген',
    style='Цепь',
    hue_order=sorted(used_genes),
    markers={'+' : 'P', '-' : 's'},
    palette='tab20',
    s=50,
    alpha=0.85
)

# biplot arrows
scaling = 11  # adjust arrow length
for i, feature in enumerate(features):
    ax.arrow(0, 0, pca.components_[0, i]*scaling, pca.components_[1, i]*scaling,
             color='gray', alpha=0.25, head_width=0.15)
    ax.text(pca.components_[0, i]*scaling*1.1, pca.components_[1, i]*scaling*1.1, feature,
            color='black', ha='center', va='center', fontsize=10, alpha=0.75)

explained = pca.explained_variance_ratio_ * 100
ax.set_xlabel(f'PC1 ({explained[0]:.1f}%)')
ax.set_ylabel(f'PC2 ({explained[1]:.1f}%)')
ax.set_xlim(-6, 8)
# ax.set_title('PCA of amino-acid composition')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0)
plt.tight_layout()
plt.show()


/tmp/ipykernel_86189/810624251.py:54: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [17]:
# plot only Mollusca
plt.figure(figsize=(8, 6))
ax = sns.scatterplot(
    data=plot_df.query('phylum == "Mollusca"'),
    x='PC1', y='PC2',
    hue='Ген',
    style='Цепь',
    hue_order=sorted(used_genes),
    markers={'+' : 'P', '-' : 's'},
    palette='tab20',
    s=50,
    alpha=0.85
)

explained = pca.explained_variance_ratio_ * 100
ax.set_xlabel(f'PC1 ({explained[0]:.1f}%)')
ax.set_ylabel(f'PC2 ({explained[1]:.1f}%)')
ax.set_xlim(-6, 8)
# ax.set_title('PCA of amino-acid composition')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0)
plt.tight_layout()
plt.show()

/tmp/ipykernel_86189/2663725772.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

# Only Mollusca, excluding ATP9
X = df_cds_flt.query('`gene name` != "ATP9" & phylum in ["Mollusca",]')\
    .set_index(['phylum', 'species', 'gene name', 'strand'])\
        .aa_sequence.apply(Counter).apply(pd.Series).drop(['X'], axis=1).fillna(0)
y = X.index.get_level_values('strand')

# Cross-validation setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro']

# Logistic regression pipeline (standardize + multinomial logistic regression)
logreg_pipe = make_pipeline(
    StandardScaler(),
    LogisticRegression(solver='lbfgs', max_iter=2000, random_state=42, fit_intercept=False)
)

# Dummy classifier (most frequent) as reference
dummy_pipe = make_pipeline(
    StandardScaler(),  # scaling doesn't affect dummy but keeps pipelines comparable
    DummyClassifier(strategy='most_frequent', random_state=42)
)

# Cross-validate both models
scores_logreg = cross_validate(logreg_pipe, X, y, cv=cv, scoring=scoring, return_train_score=False)
scores_dummy = cross_validate(dummy_pipe, X, y, cv=cv, scoring=scoring, return_train_score=False)

# Summarize CV results
def summarize_scores(scores, name):
    out = {}
    for metric in scoring:
        vals = scores[f'test_{metric}']
        out[metric] = (np.mean(vals), np.std(vals))
        print(f"{name} {metric}: {np.mean(vals):.3f} ± {np.std(vals):.3f}")
    print()
    return out

print("Cross-validation performance")
res_logreg = summarize_scores(scores_logreg, "LogReg")
res_dummy = summarize_scores(scores_dummy, "Dummy")

# Fit logistic regression on entire dataset to inspect coefficients
logreg_pipe.fit(X, y)
logreg = logreg_pipe.named_steps['logisticregression']

# Binary models store one row: log-odds of classes_[1] versus classes_[0].
coef = logreg.coef_
classes = logreg.classes_
if coef.shape[0] == 1:
    coef = np.vstack([-coef[0], coef[0]])
feature_names = list(X.columns)

# Overall importance: mean absolute coefficient across classes
mean_abs_coef = np.mean(np.abs(coef), axis=0)
feat_imp_df = pd.DataFrame({
    'feature': feature_names,
    'mean_abs_coef': mean_abs_coef
}).sort_values('mean_abs_coef', ascending=False).reset_index(drop=True)

print("Top features by mean absolute coefficient (overall importance):")
print(feat_imp_df.head(10).to_string(index=False))
print()

# Per-class top positive coefficients (features that increase log-odds for the class)
top_k = 5
for i, cls in enumerate(classes):
    class_coefs = coef[i]
    top_idx = np.argsort(class_coefs)[-top_k:][::-1]
    top_feats = [(feature_names[j], class_coefs[j]) for j in top_idx]
    print(f"Top {top_k} positive features for class '{cls}':")
    for fname, val in top_feats:
        print(f"  {fname}: {val:.4f}")
    print()
    break

# Also show top absolute contributors per class
for i, cls in enumerate(classes):
    class_coefs = coef[i]
    top_idx = np.argsort(np.abs(class_coefs))[-top_k:][::-1]
    top_feats = [(feature_names[j], class_coefs[j]) for j in top_idx]
    print(f"Top {top_k} absolute-contribution features for class '{cls}':")
    for fname, val in top_feats:
        print(f"  {fname}: {val:.4f}")
    print()
    break

Cross-validation performance
LogReg accuracy: 0.695 ± 0.046
LogReg precision_macro: 0.705 ± 0.045
LogReg recall_macro: 0.692 ± 0.046
LogReg f1_macro: 0.689 ± 0.049

Dummy accuracy: 0.520 ± 0.009
Dummy precision_macro: 0.260 ± 0.004
Dummy recall_macro: 0.500 ± 0.000
Dummy f1_macro: 0.342 ± 0.004

Top features by mean absolute coefficient (overall importance):
feature  mean_abs_coef
      W       1.085430
      Y       0.847476
      D       0.819716
      P       0.795301
      G       0.781370
      L       0.722139
      I       0.583405
      V       0.540501
      R       0.538641
      K       0.513831

Top 5 positive features for class '+':
  Y: 0.8475
  P: 0.7953
  I: 0.5834
  V: 0.5405
  R: 0.5386

Top 5 absolute-contribution features for class '+':
  W: -1.0854
  Y: 0.8475
  D: -0.8197
  P: 0.7953
  G: -0.7814



In [19]:
weights = pd.DataFrame(coef, index=classes, columns=feature_names).T
weights

,+,-
M,0.072646,-0.072646
R,0.538641,-0.538641
W,-1.085430,1.085430
L,-0.722139,0.722139
C,-0.178782,0.178782
S,-0.036201,0.036201
T,-0.162134,0.162134
N,0.002271,-0.002271
H,-0.024483,0.024483
K,0.513831,-0.513831
